# Animals interference — Polygram v0 walking tour

This notebook mirrors `examples/animals_interference.py` with explanatory cells and a final plot. It declares a 4-feature, 2-cluster Animals dictionary (`dog_poodle`, `dog_beagle`, `bird_hawk`, `bird_sparrow`) and sweeps `bird_hawk.phi` over `[0, π]` while watching the cross-cluster `(dog_poodle, bird_hawk)` overlap.

**What to look for**

- *Hierarchy preserved.* In-cluster sibling overlaps (≈ 0.8851) stay above the cross-cluster target overlap at every sweep point.
- *Single-φ does not cancel.* On this rung-1 MPS geometry, sweeping `bird_hawk.phi` alone does **not** drive `(dog_poodle, bird_hawk)` to zero — the overlap actually rises with φ, peaking near π. Driving destructive interference needs antisymmetric φ on **both** sides; that is the future `Cancellation` primitive's territory, not v0.
- *Verifiable artifact.* `experiment.materialize(...)` writes a `.q.orca.md` machine that parses + verifies clean against `q-orca>=0.7.1`.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

from examples.animals_interference import build_dictionary, build_experiment

## Build the dictionary and experiment

`build_dictionary()` returns a 4-feature `Dictionary` with two clusters (`dogs`, `birds`) and an `MPSRung1` encoding (bond-2, phase knobs enabled). `build_experiment(...)` wraps it into a phase sweep targeting the cross-cluster pair.

In [ ]:
dictionary = build_dictionary()
experiment = build_experiment(dictionary, n_points=40)

out = Path("output")
experiment.materialize(out)
result = experiment.run()

print(f"emitted {out / (experiment.name + '.q.orca.md')}")
print(f"swept {len(result.sweep_axes['bird_hawk.phi'])} points")
print(f"overlap range: [{result.overlaps.min():.4f}, {result.overlaps.max():.4f}]")

## Plot target-pair overlap vs `bird_hawk.phi`

The horizontal lines mark the published in-cluster sibling overlap (≈ 0.8851) and the φ-matched cross-cluster baseline. The sweep curve sits between them and rises with φ.

In [ ]:
phis = result.sweep_axes["bird_hawk.phi"]
overlaps = result.overlaps

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(phis, overlaps, marker="o", linewidth=1.5, label="(dog_poodle, bird_hawk)")
ax.axhline(0.8851, color="tab:green", linestyle=":", label="in-cluster siblings (0.8851)")
ax.axhline(np.cos(0.5) ** 4, color="tab:gray", linestyle=":", label=r"matched-$\varphi$ baseline ($\cos^4(0.5) \approx 0.5931$)")
ax.set_xlabel(r"$\varphi_{\mathrm{bird\_hawk}}$")
ax.set_ylabel("|<dog_poodle | bird_hawk>|")
ax.set_title("Animals interference — single-$\\varphi$ sweep on bird_hawk")
ax.set_xticks([0, np.pi / 2, np.pi])
ax.set_xticklabels(["0", r"$\pi/2$", r"$\pi$"])
ax.legend(loc="lower right")
ax.grid(alpha=0.3)
fig.tight_layout()
plt.show()

## Assertions

`hierarchical_ordering_preserved` checks that in-cluster sibling overlaps dominate the target-pair overlap at every sweep point.

In [ ]:
for name, passes in result.assertion_pass.items():
    print(f"{name}: {bool(passes.all())} (per-point pass rate: {passes.mean():.0%})")